
# HQCA CIC-IDS-2017 Zero-Day Complementarity-Shift Validation
## 10-Seed Source-Holdout GPU Experiment

This notebook is the next publication-strengthening experiment for Cyber Anathema / HQCA.

### Confirmatory question

The previous UNSW-NB15 attack-family OOD study showed that the usefulness of the
Quantum branch was not stable across attack families. In particular, the rate at
which Quantum CAT corrected Classical CAT errors changed sharply under distribution shift.

This experiment asks:

> **Does attack-dependent Classical–Quantum complementarity shift replicate under an independent CIC-IDS-2017 source/attack-family holdout protocol?**

### Pre-specified zero-day conditions

Each condition holds out the entire source CSV associated with the target attack family:

- **PortScan** → Friday afternoon PortScan source
- **Infiltration** → Thursday afternoon Infiltration source
- **Web Attack** → Thursday morning WebAttacks source

For a given condition:

1. the complete target source file is excluded from model development;
2. any occurrence of the held-out attack family in the remaining files is also removed;
3. development therefore contains benign traffic plus other known attack families;
4. the locked zero-day test contains benign traffic and the held-out attack family from the excluded source;
5. source name, `Label`, IP identifiers, Flow ID and timestamp-like identifiers are never supplied as predictive attack-family proxies.

### Models

- Classical CAT
- Quantum CAT
- 50:50 Hybrid
- Static Weighted Hybrid
- Generic RF Stacking
- Adaptive HQCA

### Primary scientific outcomes

The headline result is **not required to be HQCA superiority**. The primary outcomes are:

- `P(Q correct AND C wrong)` in development vs zero-day test;
- `P(Q correct | C wrong)` in development vs zero-day test;
- complementarity-shift ratio;
- Classical-only correction rate;
- oracle Classical-or-Quantum headroom;
- zero-day attack recall at development-controlled false-positive rates;
- 10-seed stability.

### Statistical discipline

The three held-out conditions and ten seeds are locked before inspecting results.
All conditions and all seeds must be reported. A condition must not be discarded
because it is unfavorable.


## 1. Imports, Dr. Nimbe paths and GPU preflight

In [ ]:

from pathlib import Path
import sys, json, time, platform, warnings, copy, random, math, re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, cohen_kappa_score, confusion_matrix, f1_score,
    log_loss, matthews_corrcoef, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from scipy.stats import wilcoxon

import torch
import pennylane as qml

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path(r"E:\QUANTUM_CYBER\Cyber Anathema")
CIC_DIR = (
    PROJECT_ROOT / "datasets" / "CIC-IDS-2017" / "CSV" /
    "MachineLearningCSV" / "MachineLearningCVE"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(PROJECT_ROOT)
if not CIC_DIR.exists():
    raise FileNotFoundError(CIC_DIR)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from preprocessing.clean import clean_intrusion_data
from preprocessing.feature_engineering import add_row_statistical_features
from preprocessing.scaling import build_preprocessor
from models.classical_cat import ClassicalCATClassifier

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is required. Select the Python (QUANTUM_GPU) kernel."
    )

GPU_NAME = torch.cuda.get_device_name(0)
TORCH_DEVICE = torch.device("cuda:0")

print("GPU:", GPU_NAME)
print("PennyLane:", qml.__version__)

N_QUBITS = 4
QUANTUM_LAYERS = 2
QUANTUM_BACKEND = "torch.cuda.custom_statevector"


## 2. Locked confirmatory configuration

In [ ]:

SEEDS = list(range(2026, 2036))  # exactly 10 seeds

LOCKED_ZERO_DAY_CONDITIONS = {
    "PortScan": "PortScan",
    "Infiltration": "Infilteration",
    "Web Attack": "WebAttacks",
}

# Development/test computational caps.
# Development is sampled source-aware before the final cap.
PER_SOURCE_DEV_CAP = 3000
MAX_DEV_ROWS = 12000
MAX_TEST_ATTACKS = 3000
MAX_TEST_BENIGN = 3000

# Frozen branch protocol.
OOF_FOLDS = 3
META_FOLDS = 5
QUANTUM_EPOCHS = 10
QUANTUM_BATCH_SIZE = 64
QUANTUM_PATIENCE = 3

# Operational FPR control points learned from development OOF predictions only.
FPR_TARGETS = [0.001, 0.005, 0.01]

OUTPUT_ROOT = (
    PROJECT_ROOT / "results" / "notebooks" /
    "hqca_cicids2017_zeroday_complementarity_shift_10seed_gpu"
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Output:", OUTPUT_ROOT)
print("Seeds:", SEEDS)
print("Locked zero-day conditions:", list(LOCKED_ZERO_DAY_CONDITIONS))


## 3. Native-Windows CUDA quantum implementation

In [ ]:
# -----------------------------------------------------------------------------
# Native-Windows differentiable CUDA state-vector circuit
# -----------------------------------------------------------------------------

def _ry_batch(theta):
    """Batched RY matrices for theta shape [B]."""
    c = torch.cos(theta / 2.0)
    s = torch.sin(theta / 2.0)
    gate = torch.zeros((theta.shape[0], 2, 2), dtype=torch.complex64, device=theta.device)
    gate[:, 0, 0] = c.to(torch.complex64)
    gate[:, 0, 1] = (-s).to(torch.complex64)
    gate[:, 1, 0] = s.to(torch.complex64)
    gate[:, 1, 1] = c.to(torch.complex64)
    return gate


def _rot_gate(phi, theta, omega):
    """PennyLane Rot(phi, theta, omega) matrix = RZ(omega) RY(theta) RZ(phi)."""
    c = torch.cos(theta / 2.0)
    s = torch.sin(theta / 2.0)
    a = torch.exp(-0.5j * (phi + omega)) * c
    b = -torch.exp(0.5j * (phi - omega)) * s
    c2 = torch.exp(-0.5j * (phi - omega)) * s
    d = torch.exp(0.5j * (phi + omega)) * c
    return torch.stack([torch.stack([a, b]), torch.stack([c2, d])]).to(torch.complex64)


def _apply_single_qubit(state, gate, wire, n_qubits):
    """Apply shared [2,2] or batched [B,2,2] gate to a PennyLane-ordered wire."""
    batch = state.shape[0]
    tensor = state.reshape([batch] + [2] * n_qubits)
    target_axis = 1 + int(wire)
    perm = [0] + [i for i in range(1, n_qubits + 1) if i != target_axis] + [target_axis]
    tensor = tensor.permute(perm).contiguous().reshape(batch, -1, 2)
    if gate.ndim == 2:
        tensor = torch.einsum('ij,bmj->bmi', gate, tensor)
    else:
        tensor = torch.einsum('bij,bmj->bmi', gate, tensor)
    tensor = tensor.reshape([batch] + [2] * n_qubits)
    inv = [perm.index(i) for i in range(n_qubits + 1)]
    return tensor.permute(inv).contiguous().reshape(batch, 2 ** n_qubits)


def _cnot_permutation(n_qubits, control, target, device):
    idx = torch.arange(2 ** n_qubits, device=device)
    control_mask = 1 << (n_qubits - 1 - int(control))
    target_mask = 1 << (n_qubits - 1 - int(target))
    flip = ((idx & control_mask) != 0).to(torch.long) * target_mask
    return idx ^ flip


def _z_expectation(state, wire, n_qubits):
    probs = state.abs().square()
    idx = torch.arange(2 ** n_qubits, device=state.device)
    mask = 1 << (n_qubits - 1 - int(wire))
    z = torch.where((idx & mask) == 0, 1.0, -1.0).to(probs.dtype)
    return probs @ z


class TorchCUDAStatevectorVQC(torch.nn.Module):
    """4-qubit VQC implemented entirely with differentiable PyTorch CUDA ops.

    Circuit family:
      AngleEmbedding(rotation='Y')
      -> StronglyEntanglingLayers-like blocks
         [Rot on every wire + cyclic CNOT range r=(layer mod n_qubits)+1]
      -> Z expectation on every wire
      -> linear binary-classification head
    """
    def __init__(self, n_qubits=4, n_layers=2):
        super().__init__()
        self.n_qubits = int(n_qubits)
        self.n_layers = int(n_layers)
        # PennyLane TorchLayer commonly initializes variational angles over [0, 2pi].
        self.weights = torch.nn.Parameter(
            2.0 * np.pi * torch.rand(self.n_layers, self.n_qubits, 3, dtype=torch.float32)
        )
        self.head = torch.nn.Linear(self.n_qubits, 1)

    def quantum_features(self, inputs):
        x = inputs.to(dtype=torch.float32)
        batch = x.shape[0]
        state = torch.zeros(
            (batch, 2 ** self.n_qubits), dtype=torch.complex64, device=x.device
        )
        state[:, 0] = 1.0 + 0.0j

        # PennyLane AngleEmbedding(..., rotation='Y').
        for wire in range(self.n_qubits):
            state = _apply_single_qubit(
                state, _ry_batch(x[:, wire]), wire, self.n_qubits
            )

        # PennyLane StronglyEntanglingLayers default ranges: 1, 2, ..., cycling.
        for layer in range(self.n_layers):
            for wire in range(self.n_qubits):
                phi, theta, omega = self.weights[layer, wire]
                gate = _rot_gate(phi, theta, omega)
                state = _apply_single_qubit(state, gate, wire, self.n_qubits)

            r = (layer % self.n_qubits) + 1
            # r divisible by n_qubits would create self-CNOT; PennyLane does not use that
            # for valid StronglyEntanglingLayers ranges. With 2 layers / 4 qubits: r=1,2.
            for control in range(self.n_qubits):
                target = (control + r) % self.n_qubits
                if target == control:
                    continue
                perm = _cnot_permutation(self.n_qubits, control, target, state.device)
                state = state[:, perm]

        features = torch.stack(
            [_z_expectation(state, wire, self.n_qubits) for wire in range(self.n_qubits)],
            dim=1,
        )
        return features.to(torch.float32)

    def forward(self, inputs):
        qfeatures = self.quantum_features(inputs)
        return self.head(qfeatures).reshape(-1)


class GPUQuantumCATClassifier(BaseEstimator, ClassifierMixin):
    """GPU-enforced QCAT classifier using the native-Windows CUDA VQC above."""
    def __init__(
        self, n_qubits=4, n_layers=2, epochs=16, batch_size=64,
        learning_rate=0.01, weight_decay=1e-4, hard_sample_weight=2.0,
        validation_fraction=0.15, patience=4, random_state=2026, verbose=False,
    ):
        self.n_qubits=n_qubits; self.n_layers=n_layers; self.epochs=epochs
        self.batch_size=batch_size; self.learning_rate=learning_rate
        self.weight_decay=weight_decay; self.hard_sample_weight=hard_sample_weight
        self.validation_fraction=validation_fraction; self.patience=patience
        self.random_state=random_state; self.verbose=verbose

    @staticmethod
    def _dense(X):
        return X.toarray() if hasattr(X,'toarray') else np.asarray(X)

    def _prepare_features(self, X, fit=False):
        arr=np.asarray(self._dense(X), dtype=np.float32)
        if fit:
            n_components=min(self.n_qubits, arr.shape[1], max(1, arr.shape[0]-1))
            self.pca_=PCA(n_components=n_components, random_state=self.random_state)
            reduced=self.pca_.fit_transform(arr)
            if n_components<self.n_qubits:
                reduced=np.pad(reduced,((0,0),(0,self.n_qubits-n_components)))
            self.angle_scaler_=MinMaxScaler(feature_range=(-np.pi,np.pi))
            return self.angle_scaler_.fit_transform(reduced).astype(np.float32)
        reduced=self.pca_.transform(arr)
        if reduced.shape[1]<self.n_qubits:
            reduced=np.pad(reduced,((0,0),(0,self.n_qubits-reduced.shape[1])))
        return self.angle_scaler_.transform(reduced).astype(np.float32)

    def fit(self,X,y):
        seed_everything(self.random_state)
        features=self._prepare_features(X,fit=True)
        labels=np.asarray(y,dtype=np.float32)
        if not set(np.unique(labels)).issubset({0.0,1.0}):
            raise ValueError('GPUQuantumCATClassifier supports binary labels 0/1.')

        idx=np.arange(len(labels)); rng=np.random.default_rng(self.random_state); rng.shuffle(idx)
        n_val=max(1,int(len(idx)*self.validation_fraction)); val_idx,train_idx=idx[:n_val],idx[n_val:]
        Xtr=torch.tensor(features[train_idx],dtype=torch.float32,device=TORCH_DEVICE)
        ytr=torch.tensor(labels[train_idx],dtype=torch.float32,device=TORCH_DEVICE)
        Xva=torch.tensor(features[val_idx],dtype=torch.float32,device=TORCH_DEVICE)
        yva=torch.tensor(labels[val_idx],dtype=torch.float32,device=TORCH_DEVICE)

        self.model_=TorchCUDAStatevectorVQC(self.n_qubits,self.n_layers).to(TORCH_DEVICE)
        opt=torch.optim.Adam(self.model_.parameters(),lr=self.learning_rate,weight_decay=self.weight_decay)
        loss_fn=torch.nn.BCEWithLogitsLoss(reduction='none')
        best=float('inf'); best_state=None; stale=0

        for epoch in range(self.epochs):
            order=rng.permutation(len(train_idx)); self.model_.train()
            for start in range(0,len(order),self.batch_size):
                b=order[start:start+self.batch_size]
                bx=Xtr[b]; by=ytr[b]
                opt.zero_grad(set_to_none=True)
                logits=self.model_(bx).reshape(-1)
                prob=torch.sigmoid(logits).detach()
                uncertainty=1.0-torch.abs(2.0*prob-1.0)
                errors=((prob>=0.5).float()!=by).float()
                sample_weights=1.0+self.hard_sample_weight*(uncertainty+errors)
                loss=(loss_fn(logits,by)*sample_weights).mean()
                loss.backward(); opt.step()

            self.model_.eval()
            with torch.no_grad():
                v_logits=self.model_(Xva).reshape(-1)
                v_loss=torch.nn.functional.binary_cross_entropy_with_logits(v_logits,yva).item()
            if self.verbose:
                print(f'epoch={epoch+1} validation_loss={v_loss:.6f}')
            if v_loss<best-1e-5:
                best=v_loss
                best_state={k:v.detach().cpu().clone() for k,v in self.model_.state_dict().items()}
                stale=0
            else:
                stale+=1
                if stale>=self.patience:
                    break

        if best_state is not None:
            self.model_.load_state_dict(best_state)
        self.model_.to(TORCH_DEVICE)
        self.classes_=np.array([0,1]); self.n_features_in_=self._dense(X).shape[1]
        self.gpu_backend_=QUANTUM_BACKEND; self.gpu_name_=GPU_NAME
        self.best_validation_loss_=best
        return self

    def predict_proba(self,X):
        check_is_fitted(self,['model_','pca_','angle_scaler_'])
        features=torch.tensor(self._prepare_features(X),dtype=torch.float32,device=TORCH_DEVICE)
        self.model_.eval()
        outputs=[]
        with torch.no_grad():
            for start in range(0,len(features),max(1,self.batch_size*4)):
                logits=self.model_(features[start:start+max(1,self.batch_size*4)])
                outputs.append(torch.sigmoid(logits).detach().cpu())
        positive=torch.cat(outputs).numpy()
        return np.column_stack([1.0-positive,positive])

    def predict(self,X):
        return (self.predict_proba(X)[:,1]>=0.5).astype(int)


## 4. Mandatory CUDA/PennyLane circuit-equivalence audit

In [ ]:
# Forward-equivalence test before any ToN-IoT model fitting.
AUDIT_SAMPLES = 8
AUDIT_TOLERANCE = 1e-5

audit_rng = np.random.default_rng(RANDOM_STATE + 77)
audit_x_np = audit_rng.uniform(-np.pi, np.pi, size=(AUDIT_SAMPLES, N_QUBITS)).astype(np.float32)
audit_w_np = audit_rng.uniform(0.0, 2*np.pi, size=(QUANTUM_LAYERS, N_QUBITS, 3)).astype(np.float32)

# Native CUDA features.
audit_model = TorchCUDAStatevectorVQC(N_QUBITS, QUANTUM_LAYERS).to(TORCH_DEVICE)
with torch.no_grad():
    audit_model.weights.copy_(torch.tensor(audit_w_np, dtype=torch.float32, device=TORCH_DEVICE))
    cuda_features = audit_model.quantum_features(
        torch.tensor(audit_x_np, dtype=torch.float32, device=TORCH_DEVICE)
    ).detach().cpu().numpy()

# PennyLane CPU reference features using the original circuit family.
pl_dev = qml.device('default.qubit', wires=N_QUBITS)
@qml.qnode(pl_dev)
def reference_qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation='Y')
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

pl_features = np.vstack([
    np.asarray(reference_qnode(audit_x_np[i], audit_w_np), dtype=float)
    for i in range(AUDIT_SAMPLES)
])

abs_diff = np.abs(cuda_features - pl_features)
CIRCUIT_EQUIVALENCE_MAX_ABS_ERROR = float(abs_diff.max())
CIRCUIT_EQUIVALENCE_MEAN_ABS_ERROR = float(abs_diff.mean())
CIRCUIT_EQUIVALENCE_PASSED = bool(CIRCUIT_EQUIVALENCE_MAX_ABS_ERROR < AUDIT_TOLERANCE)

print('CUDA circuit device:', TORCH_DEVICE)
print('Max absolute error:', CIRCUIT_EQUIVALENCE_MAX_ABS_ERROR)
print('Mean absolute error:', CIRCUIT_EQUIVALENCE_MEAN_ABS_ERROR)
print('Tolerance:', AUDIT_TOLERANCE)
print('Equivalence passed:', CIRCUIT_EQUIVALENCE_PASSED)

if not CIRCUIT_EQUIVALENCE_PASSED:
    raise AssertionError(
        'Native CUDA circuit failed the PennyLane forward-equivalence audit. '
        'Do not run the prospective experiment until the circuit implementation is corrected.'
    )


## 5. CIC-IDS-2017 source audit and zero-day label normalization

In [ ]:

csv_files = sorted(CIC_DIR.glob("*.csv"))
if len(csv_files) != 8:
    print(f"WARNING: expected 8 CIC-IDS-2017 CSVs, found {len(csv_files)}")

source_audit_rows = []

def normalize_columns(frame):
    frame = frame.copy()
    frame.columns = [str(c).strip() for c in frame.columns]
    return frame

def locate_label_column(frame):
    for col in frame.columns:
        if str(col).strip().lower() == "label":
            return col
    raise KeyError("Label column not found.")

def normalize_attack_family(label):
    s = str(label).strip()
    low = s.lower()

    if low == "benign":
        return "BENIGN"
    if "portscan" in low:
        return "PortScan"
    if "infiltration" in low or "infilteration" in low:
        return "Infiltration"
    if "web attack" in low:
        return "Web Attack"
    if "ddos" in low:
        return "DDoS"
    if low == "bot" or "bot" in low:
        return "Bot"
    if "ftp-patator" in low:
        return "FTP-Patator"
    if "ssh-patator" in low:
        return "SSH-Patator"
    if "heartbleed" in low:
        return "Heartbleed"
    if "dos" in low:
        return "DoS"
    return s

def source_matches_token(path, token):
    return token.lower() in path.name.lower()

for path in csv_files:
    # Only read Label to make the audit inexpensive.
    probe = pd.read_csv(path, usecols=lambda c: str(c).strip().lower() == "label", low_memory=False)
    probe = normalize_columns(probe)
    label_col = locate_label_column(probe)
    families = probe[label_col].map(normalize_attack_family)
    counts = families.value_counts(dropna=False)

    source_audit_rows.append({
        "source_file": path.name,
        "rows": len(probe),
        "benign": int(counts.get("BENIGN", 0)),
        "PortScan": int(counts.get("PortScan", 0)),
        "Infiltration": int(counts.get("Infiltration", 0)),
        "Web Attack": int(counts.get("Web Attack", 0)),
        "unique_normalized_families": int(counts.shape[0]),
    })

source_audit = pd.DataFrame(source_audit_rows)
display(source_audit)
source_audit.to_csv(OUTPUT_ROOT / "cicids_source_audit.csv", index=False)

resolved_sources = {}
for family, token in LOCKED_ZERO_DAY_CONDITIONS.items():
    matches = [p for p in csv_files if source_matches_token(p, token)]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one source file for {family}/{token}; got {[m.name for m in matches]}"
        )
    resolved_sources[family] = matches[0]

print("\nResolved locked test sources:")
for family, path in resolved_sources.items():
    print(f"  {family}: {path.name}")

pd.DataFrame([
    {"heldout_family": k, "heldout_source": v.name}
    for k, v in resolved_sources.items()
]).to_csv(OUTPUT_ROOT / "locked_zero_day_sources.csv", index=False)


## 6. Memory-conscious source-aware sampling and leakage controls

In [ ]:

IDENTIFIER_PATTERNS = [
    r"^flow id$", r"^source ip$", r"^destination ip$",
    r"^src ip$", r"^dst ip$", r"timestamp",
]

def drop_explicit_identifiers(frame):
    drop_cols = []
    for col in frame.columns:
        low = str(col).strip().lower()
        if any(re.search(pattern, low) for pattern in IDENTIFIER_PATTERNS):
            drop_cols.append(col)
    return frame.drop(columns=drop_cols, errors="ignore"), drop_cols

def deterministic_stratified_cap(frame, label_col, max_rows, seed):
    if len(frame) <= max_rows:
        return frame.reset_index(drop=True)

    y = frame[label_col]
    if y.nunique() < 2:
        return frame.sample(n=max_rows, random_state=seed).reset_index(drop=True)

    idx = np.arange(len(frame))
    keep, _ = train_test_split(
        idx,
        train_size=max_rows,
        stratify=y,
        random_state=seed,
    )
    return frame.iloc[keep].reset_index(drop=True)

def read_source_for_development(path, heldout_family, cap, seed):
    df = pd.read_csv(path, low_memory=False)
    df = normalize_columns(df)
    label_col = locate_label_column(df)

    family = df[label_col].map(normalize_attack_family)

    # Remove held-out attack family anywhere in development.
    keep = family != heldout_family
    df = df.loc[keep].copy()
    family = family.loc[keep]

    df["__binary_label__"] = (family != "BENIGN").astype(int).to_numpy()
    df["__source_file__"] = path.name

    # Label is removed before modeling; source is kept only for audit until cleaning.
    df = df.drop(columns=[label_col], errors="ignore")

    df = deterministic_stratified_cap(
        df, "__binary_label__", min(cap, len(df)), seed
    )
    return df

def read_zero_day_test(path, heldout_family, seed):
    df = pd.read_csv(path, low_memory=False)
    df = normalize_columns(df)
    label_col = locate_label_column(df)

    family = df[label_col].map(normalize_attack_family)

    # Isolated zero-day binary test: benign + target unseen family only.
    keep = (family == "BENIGN") | (family == heldout_family)
    df = df.loc[keep].copy()
    family = family.loc[keep].reset_index(drop=True)
    df = df.reset_index(drop=True)

    attack_idx = np.where(family.to_numpy() == heldout_family)[0]
    benign_idx = np.where(family.to_numpy() == "BENIGN")[0]

    rng = np.random.default_rng(seed)
    if len(attack_idx) > MAX_TEST_ATTACKS:
        attack_idx = rng.choice(attack_idx, size=MAX_TEST_ATTACKS, replace=False)
    if len(benign_idx) > MAX_TEST_BENIGN:
        benign_idx = rng.choice(benign_idx, size=MAX_TEST_BENIGN, replace=False)

    keep_idx = np.sort(np.concatenate([attack_idx, benign_idx]))
    df = df.iloc[keep_idx].reset_index(drop=True)
    family = family.iloc[keep_idx].reset_index(drop=True)

    df["__binary_label__"] = (family != "BENIGN").astype(int).to_numpy()
    df["__source_file__"] = path.name
    df["__attack_family_audit__"] = family.to_numpy()

    df = df.drop(columns=[label_col], errors="ignore")
    return df

def clean_model_frame(frame):
    audit_source = frame["__source_file__"].copy() if "__source_file__" in frame else None
    audit_family = frame["__attack_family_audit__"].copy() if "__attack_family_audit__" in frame else None

    model = frame.drop(
        columns=["__source_file__", "__attack_family_audit__"],
        errors="ignore",
    ).copy()
    model, explicit_dropped = drop_explicit_identifiers(model)

    cleaned, normalized_label, report = clean_intrusion_data(
        model,
        "__binary_label__",
        binary_target=True,
        drop_duplicates=True,
        remove_identifiers=True,
        drop_constants=False,
    )
    cleaned, generated = add_row_statistical_features(
        cleaned,
        exclude=[normalized_label],
    )

    y = cleaned[normalized_label].astype(int).reset_index(drop=True)
    X = cleaned.drop(columns=[normalized_label], errors="ignore").reset_index(drop=True)

    forbidden = {
        "label", "attack_cat", "attack_category", "__source_file__",
        "__attack_family_audit__",
    }
    overlap = forbidden.intersection({str(c).strip().lower() for c in X.columns})
    if overlap:
        raise AssertionError(f"Leakage-control failure: {overlap}")

    return X, y, report, generated, explicit_dropped

def build_shift_dataset(heldout_family, heldout_source):
    data_seed = 45000 + sum(ord(c) for c in heldout_family)

    dev_parts = []
    for i, path in enumerate(csv_files):
        if path.resolve() == heldout_source.resolve():
            continue
        part = read_source_for_development(
            path,
            heldout_family=heldout_family,
            cap=PER_SOURCE_DEV_CAP,
            seed=data_seed + i,
        )
        dev_parts.append(part)

    dev_raw = pd.concat(dev_parts, ignore_index=True)
    dev_raw = deterministic_stratified_cap(
        dev_raw,
        "__binary_label__",
        min(MAX_DEV_ROWS, len(dev_raw)),
        data_seed + 100,
    )

    test_raw = read_zero_day_test(
        heldout_source,
        heldout_family=heldout_family,
        seed=data_seed + 200,
    )

    X_dev, y_dev, *_ = clean_model_frame(dev_raw)
    X_test, y_test, *_ = clean_model_frame(test_raw)

    # Strict schema alignment using the development feature schema.
    X_test = X_test.reindex(columns=X_dev.columns)

    audit = pd.DataFrame([
        {
            "heldout_family": heldout_family,
            "split": "development",
            "source_rule": "all non-heldout source CSVs",
            "rows": len(y_dev),
            "benign": int((y_dev == 0).sum()),
            "attack": int((y_dev == 1).sum()),
        },
        {
            "heldout_family": heldout_family,
            "split": "zero_day_test",
            "source_rule": heldout_source.name,
            "rows": len(y_test),
            "benign": int((y_test == 0).sum()),
            "attack": int((y_test == 1).sum()),
        },
    ])
    return X_dev, y_dev, X_test, y_test, audit

dataset_cache = {}
audits = []

for family, source in resolved_sources.items():
    built = build_shift_dataset(family, source)
    dataset_cache[family] = built
    audits.append(built[4])

split_audit = pd.concat(audits, ignore_index=True)
display(split_audit)
split_audit.to_csv(OUTPUT_ROOT / "zero_day_split_audit.csv", index=False)


## 7. Classical/Quantum branches and nested leakage-safe hybrid fusion

In [ ]:

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def make_classical_branch(reference_X, seed):
    pre = build_preprocessor(
        reference_X,
        scaler="robust",
        one_hot_min_frequency=2,
    ).transformer

    clf = ClassicalCATClassifier(
        adaptation_rounds=4,
        uncertainty_weight=1.5,
        error_weight=2.0,
        memory_fraction=0.10,
        replay_weight=2.5,
        random_state=seed,
    )
    return Pipeline([("preprocessor", pre), ("classifier", clf)])

def make_quantum_branch(reference_X, seed):
    pre = build_preprocessor(
        reference_X,
        scaler="minmax",
        one_hot_min_frequency=2,
    ).transformer

    clf = GPUQuantumCATClassifier(
        n_qubits=N_QUBITS,
        n_layers=QUANTUM_LAYERS,
        epochs=QUANTUM_EPOCHS,
        batch_size=QUANTUM_BATCH_SIZE,
        learning_rate=0.01,
        weight_decay=1e-4,
        hard_sample_weight=2.0,
        patience=QUANTUM_PATIENCE,
        random_state=seed,
        verbose=False,
    )
    return Pipeline([("preprocessor", pre), ("classifier", clf)])

def positive_probability(model, X):
    return np.asarray(model.predict_proba(X)[:, 1], dtype=float)

def binary_entropy(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1 - 1e-12)
    return -(p*np.log2(p) + (1-p)*np.log2(1-p))

def reflexive_features(pc, pq):
    pc = np.asarray(pc, dtype=float)
    pq = np.asarray(pq, dtype=float)
    return pd.DataFrame({
        "p_classical": pc,
        "p_quantum": pq,
        "mean_probability": 0.5*(pc + pq),
        "branch_interaction": pc*pq,
        "branch_disagreement": np.abs(pc-pq),
        "classical_entropy": binary_entropy(pc),
        "quantum_entropy": binary_entropy(pq),
    })

def probability_metrics(y_true, p, threshold):
    y = np.asarray(y_true, dtype=int)
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1-1e-12)
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()

    try:
        roc = roc_auc_score(y, p)
    except ValueError:
        roc = np.nan
    try:
        pr = average_precision_score(y, p)
    except ValueError:
        pr = np.nan

    return {
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "specificity": tn / max(tn + fp, 1),
        "f1": f1_score(y, pred, zero_division=0),
        "mcc": matthews_corrcoef(y, pred),
        "kappa": cohen_kappa_score(y, pred),
        "roc_auc": roc,
        "pr_auc": pr,
        "log_loss": log_loss(y, p, labels=[0, 1]),
        "brier_score": brier_score_loss(y, p),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def optimize_threshold(y, p):
    best = None
    for t in np.linspace(0.02, 0.98, 193):
        m = probability_metrics(y, p, t)
        key = (m["mcc"], m["balanced_accuracy"], m["f1"])
        if best is None or key > best[0]:
            best = (key, float(t), m)
    return best[1], best[2]

def generate_branch_oof(X, y, seed):
    min_class = int(pd.Series(y).value_counts().min())
    n_splits = max(2, min(OOF_FOLDS, min_class))
    splitter = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )

    pc = np.full(len(y), np.nan)
    pq = np.full(len(y), np.nan)
    fold_id = np.full(len(y), -1)
    fold_times = []

    for fold, (tr, va) in enumerate(splitter.split(X, y), start=1):
        Xtr, Xva = X.iloc[tr], X.iloc[va]
        ytr = y.iloc[tr]
        fold_seed = seed + fold

        classical = make_classical_branch(Xtr, fold_seed)
        t0 = time.perf_counter()
        classical.fit(Xtr, ytr)
        csec = time.perf_counter() - t0
        pc[va] = positive_probability(classical, Xva)

        quantum = make_quantum_branch(Xtr, fold_seed)
        t0 = time.perf_counter()
        quantum.fit(Xtr, ytr)
        qsec = time.perf_counter() - t0
        pq[va] = positive_probability(quantum, Xva)

        fold_id[va] = fold
        fold_times.append({
            "fold": fold,
            "classical_seconds": csec,
            "quantum_seconds": qsec,
        })

        print(
            f"seed={seed} fold={fold}/{n_splits} "
            f"classical={csec:.1f}s quantum={qsec:.1f}s"
        )

    if np.isnan(pc).any() or np.isnan(pq).any():
        raise RuntimeError("Incomplete branch OOF predictions.")

    return pd.DataFrame({
        "y_true": y.to_numpy(dtype=int),
        "fold": fold_id.astype(int),
        "p_classical": pc,
        "p_quantum": pq,
    }), pd.DataFrame(fold_times)

def cross_fitted_meta_probabilities(features, y, model_factory, seed):
    y = np.asarray(y, dtype=int)
    min_class = int(pd.Series(y).value_counts().min())
    n_splits = max(2, min(META_FOLDS, min_class))

    splitter = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed + 200,
    )
    pred = np.full(len(y), np.nan)

    for fold, (tr, va) in enumerate(splitter.split(features, y), start=1):
        model = model_factory(seed + fold)
        model.fit(features.iloc[tr], y[tr])
        pred[va] = model.predict_proba(features.iloc[va])[:, 1]

    if np.isnan(pred).any():
        raise RuntimeError("Incomplete meta-OOF predictions.")
    return pred

def lr_meta_factory(seed):
    return LogisticRegression(
        solver="liblinear",
        C=1.0,
        class_weight="balanced",
        max_iter=2000,
        random_state=seed,
    )

def rf_meta_factory(seed):
    return RandomForestClassifier(
        n_estimators=350,
        max_depth=5,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=seed,
        n_jobs=-1,
    )

def fit_fusion_specs(oof, seed):
    y = oof["y_true"].to_numpy(dtype=int)
    pc = oof["p_classical"].to_numpy(float)
    pq = oof["p_quantum"].to_numpy(float)

    specs = {}
    dev_probabilities = {}

    tc, _ = optimize_threshold(y, pc)
    tq, _ = optimize_threshold(y, pq)
    specs["Classical CAT"] = {"kind": "classical", "threshold": tc}
    specs["Quantum CAT"] = {"kind": "quantum", "threshold": tq}
    dev_probabilities["Classical CAT"] = pc
    dev_probabilities["Quantum CAT"] = pq

    meanp = 0.5*(pc + pq)
    tm, _ = optimize_threshold(y, meanp)
    specs["50:50 Hybrid"] = {"kind": "mean", "threshold": tm}
    dev_probabilities["50:50 Hybrid"] = meanp

    best = None
    for w in np.linspace(0, 1, 41):
        p = w*pc + (1-w)*pq
        t, m = optimize_threshold(y, p)
        key = (m["mcc"], m["balanced_accuracy"], m["f1"])
        if best is None or key > best["key"]:
            best = {"key": key, "w": float(w), "threshold": float(t), "p": p}

    specs["Static Weighted Hybrid"] = {
        "kind": "static",
        "classical_weight": best["w"],
        "quantum_weight": 1-best["w"],
        "threshold": best["threshold"],
    }
    dev_probabilities["Static Weighted Hybrid"] = best["p"]

    raw = pd.DataFrame({"p_classical": pc, "p_quantum": pq})
    rf_meta_oof = cross_fitted_meta_probabilities(raw, y, rf_meta_factory, seed)
    trf, _ = optimize_threshold(y, rf_meta_oof)
    rf_final = rf_meta_factory(seed + 700)
    rf_final.fit(raw, y)

    specs["Generic RF Stacking"] = {
        "kind": "stack",
        "model": rf_final,
        "threshold": trf,
        "feature_names": raw.columns.tolist(),
    }
    dev_probabilities["Generic RF Stacking"] = rf_meta_oof

    hq = reflexive_features(pc, pq)
    hq_meta_oof = cross_fitted_meta_probabilities(hq, y, lr_meta_factory, seed + 1000)
    thq, _ = optimize_threshold(y, hq_meta_oof)
    hq_final = lr_meta_factory(seed + 1700)
    hq_final.fit(hq, y)

    specs["Adaptive HQCA"] = {
        "kind": "adaptive",
        "model": hq_final,
        "threshold": thq,
        "feature_names": hq.columns.tolist(),
    }
    dev_probabilities["Adaptive HQCA"] = hq_meta_oof

    return specs, dev_probabilities

def train_system(X, y, seed):
    seed_everything(seed)
    started = time.perf_counter()

    oof, fold_times = generate_branch_oof(X, y, seed)
    specs, dev_probabilities = fit_fusion_specs(oof, seed)

    classical = make_classical_branch(X, seed + 5000)
    t0 = time.perf_counter()
    classical.fit(X, y)
    csec = time.perf_counter() - t0

    quantum = make_quantum_branch(X, seed + 5000)
    t0 = time.perf_counter()
    quantum.fit(X, y)
    qsec = time.perf_counter() - t0

    return {
        "oof": oof,
        "fold_times": fold_times,
        "specs": specs,
        "dev_probabilities": dev_probabilities,
        "classical_model": classical,
        "quantum_model": quantum,
        "final_classical_seconds": csec,
        "final_quantum_seconds": qsec,
        "total_seconds": time.perf_counter() - started,
    }

def evaluate_system(system, X, y):
    pc = positive_probability(system["classical_model"], X)
    pq = positive_probability(system["quantum_model"], X)

    raw = pd.DataFrame({"p_classical": pc, "p_quantum": pq})
    hq = reflexive_features(pc, pq)

    rows = []
    predictions = {}

    for name, spec in system["specs"].items():
        if spec["kind"] == "classical":
            p = pc
        elif spec["kind"] == "quantum":
            p = pq
        elif spec["kind"] == "mean":
            p = 0.5*(pc + pq)
        elif spec["kind"] == "static":
            p = (
                spec["classical_weight"]*pc +
                spec["quantum_weight"]*pq
            )
        elif spec["kind"] == "stack":
            p = spec["model"].predict_proba(
                raw.reindex(columns=spec["feature_names"])
            )[:, 1]
        elif spec["kind"] == "adaptive":
            p = spec["model"].predict_proba(
                hq.reindex(columns=spec["feature_names"])
            )[:, 1]
        else:
            raise ValueError(spec["kind"])

        threshold = float(spec["threshold"])
        m = probability_metrics(y, p, threshold)
        rows.append({"model": name, "threshold": threshold, **m})

        predictions[name] = pd.DataFrame({
            "y_true": np.asarray(y, dtype=int),
            "y_probability": p,
            "y_pred": (p >= threshold).astype(int),
            "p_classical": pc,
            "p_quantum": pq,
        })

    return pd.DataFrame(rows), predictions


## 8. Complementarity-shift and controlled-FPR analysis

In [ ]:

def complementarity_stats(y, pc, pq, tc, tq):
    y = np.asarray(y, dtype=int)
    cp = (np.asarray(pc) >= tc).astype(int)
    qp = (np.asarray(pq) >= tq).astype(int)

    c_correct = cp == y
    q_correct = qp == y

    q_only = (~c_correct) & q_correct
    c_only = c_correct & (~q_correct)
    both_correct = c_correct & q_correct
    both_wrong = (~c_correct) & (~q_correct)

    classical_errors = int((~c_correct).sum())
    quantum_errors = int((~q_correct).sum())

    return {
        "n": len(y),
        "both_correct": int(both_correct.sum()),
        "quantum_only_correct": int(q_only.sum()),
        "classical_only_correct": int(c_only.sum()),
        "both_wrong": int(both_wrong.sum()),
        "quantum_only_rate_all": float(q_only.mean()),
        "classical_only_rate_all": float(c_only.mean()),
        "quantum_rescue_given_classical_error": float(
            q_only.sum() / max(classical_errors, 1)
        ),
        "classical_rescue_given_quantum_error": float(
            c_only.sum() / max(quantum_errors, 1)
        ),
        "classical_accuracy": float(c_correct.mean()),
        "quantum_accuracy": float(q_correct.mean()),
        "oracle_accuracy": float((c_correct | q_correct).mean()),
        "classical_errors": classical_errors,
        "quantum_errors": quantum_errors,
    }

def dev_fpr_threshold(y_dev, p_dev, target_fpr):
    y_dev = np.asarray(y_dev, dtype=int)
    p_dev = np.asarray(p_dev, dtype=float)
    benign_scores = p_dev[y_dev == 0]
    if len(benign_scores) == 0:
        return np.nan

    q = 1.0 - target_fpr
    try:
        threshold = np.quantile(
            benign_scores, q, method="higher"
        )
    except TypeError:
        threshold = np.quantile(
            benign_scores, q, interpolation="higher"
        )
    return float(threshold)

def controlled_fpr_metrics(y_test, p_test, threshold):
    y = np.asarray(y_test, dtype=int)
    p = np.asarray(p_test, dtype=float)
    pred = (p >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "realized_test_fpr": fp / max(fp + tn, 1),
        "zero_day_recall": tp / max(tp + fn, 1),
        "precision": tp / max(tp + fp, 1),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
    }

def shift_ratio(test_value, dev_value):
    if dev_value <= 0:
        return np.nan
    return float(test_value / dev_value)


## 9. Locked 10-seed zero-day experiment

In [ ]:

model_rows = []
complement_rows = []
fpr_rows = []
runtime_rows = []
fusion_weight_rows = []

for heldout_family in LOCKED_ZERO_DAY_CONDITIONS:
    print("\n" + "="*100)
    print("ZERO-DAY CONDITION:", heldout_family)
    print("HELD-OUT SOURCE:", resolved_sources[heldout_family].name)
    print("="*100)

    X_dev, y_dev, X_test, y_test, _ = dataset_cache[heldout_family]

    condition_dir = OUTPUT_ROOT / heldout_family.replace(" ", "_")
    condition_dir.mkdir(parents=True, exist_ok=True)

    for seed in SEEDS:
        print(f"\n--- {heldout_family} | seed={seed} ---")
        run_dir = condition_dir / f"seed_{seed}"
        run_dir.mkdir(parents=True, exist_ok=True)

        system = train_system(X_dev, y_dev, seed)
        results, predictions = evaluate_system(system, X_test, y_test)

        results.insert(0, "seed", seed)
        results.insert(0, "heldout_family", heldout_family)
        model_rows.append(results)

        results.to_csv(run_dir / "zero_day_model_metrics.csv", index=False)
        system["oof"].to_csv(run_dir / "development_branch_oof.csv", index=False)
        system["fold_times"].to_csv(run_dir / "fold_times.csv", index=False)

        # Central branch complementarity shift.
        tc = system["specs"]["Classical CAT"]["threshold"]
        tq = system["specs"]["Quantum CAT"]["threshold"]

        dev_comp = complementarity_stats(
            system["oof"]["y_true"],
            system["oof"]["p_classical"],
            system["oof"]["p_quantum"],
            tc, tq,
        )

        test_pc = predictions["Classical CAT"]["p_classical"].to_numpy(float)
        test_pq = predictions["Classical CAT"]["p_quantum"].to_numpy(float)

        test_comp = complementarity_stats(
            y_test, test_pc, test_pq, tc, tq,
        )

        comp_row = {
            "heldout_family": heldout_family,
            "seed": seed,
            "dev_quantum_only_rate_all": dev_comp["quantum_only_rate_all"],
            "test_quantum_only_rate_all": test_comp["quantum_only_rate_all"],
            "quantum_only_shift_ratio": shift_ratio(
                test_comp["quantum_only_rate_all"],
                dev_comp["quantum_only_rate_all"],
            ),
            "dev_quantum_rescue_given_classical_error":
                dev_comp["quantum_rescue_given_classical_error"],
            "test_quantum_rescue_given_classical_error":
                test_comp["quantum_rescue_given_classical_error"],
            "conditional_quantum_rescue_shift_ratio": shift_ratio(
                test_comp["quantum_rescue_given_classical_error"],
                dev_comp["quantum_rescue_given_classical_error"],
            ),
            "dev_classical_only_rate_all": dev_comp["classical_only_rate_all"],
            "test_classical_only_rate_all": test_comp["classical_only_rate_all"],
            "classical_only_shift_ratio": shift_ratio(
                test_comp["classical_only_rate_all"],
                dev_comp["classical_only_rate_all"],
            ),
            "test_oracle_accuracy": test_comp["oracle_accuracy"],
            "test_classical_accuracy": test_comp["classical_accuracy"],
            "test_quantum_accuracy": test_comp["quantum_accuracy"],
            "oracle_headroom_over_classical":
                test_comp["oracle_accuracy"] - test_comp["classical_accuracy"],
            "test_quantum_only_correct": test_comp["quantum_only_correct"],
            "test_classical_only_correct": test_comp["classical_only_correct"],
            "test_both_wrong": test_comp["both_wrong"],
        }
        complement_rows.append(comp_row)

        # Development-controlled FPR thresholds for every probabilistic model.
        y_dev_oof = system["oof"]["y_true"].to_numpy(int)

        for model_name, dev_p in system["dev_probabilities"].items():
            test_p = predictions[model_name]["y_probability"].to_numpy(float)

            for target_fpr in FPR_TARGETS:
                threshold = dev_fpr_threshold(
                    y_dev_oof, dev_p, target_fpr
                )
                cf = controlled_fpr_metrics(y_test, test_p, threshold)
                fpr_rows.append({
                    "heldout_family": heldout_family,
                    "seed": seed,
                    "model": model_name,
                    "development_target_fpr": target_fpr,
                    "development_threshold": threshold,
                    **cf,
                })

        sw = system["specs"]["Static Weighted Hybrid"]
        fusion_weight_rows.append({
            "heldout_family": heldout_family,
            "seed": seed,
            "classical_weight": sw["classical_weight"],
            "quantum_weight": sw["quantum_weight"],
        })

        runtime_rows.append({
            "heldout_family": heldout_family,
            "seed": seed,
            "total_seconds": system["total_seconds"],
            "final_classical_seconds": system["final_classical_seconds"],
            "final_quantum_seconds": system["final_quantum_seconds"],
        })

        # Save per-example probabilities for reproducibility.
        save_pred = pd.DataFrame({
            "y_true": np.asarray(y_test, dtype=int),
            "p_classical": test_pc,
            "p_quantum": test_pq,
            "classical_pred": predictions["Classical CAT"]["y_pred"],
            "quantum_pred": predictions["Quantum CAT"]["y_pred"],
            "adaptive_hqca_probability":
                predictions["Adaptive HQCA"]["y_probability"],
            "rf_stack_probability":
                predictions["Generic RF Stacking"]["y_probability"],
        })
        save_pred.to_csv(run_dir / "zero_day_predictions.csv", index=False)

        print(
            results[
                ["model", "mcc", "balanced_accuracy", "recall", "specificity", "roc_auc"]
            ].sort_values("mcc", ascending=False).to_string(index=False)
        )
        print(
            "Quantum rescue shift:",
            f'{comp_row["dev_quantum_rescue_given_classical_error"]:.4f}',
            "->",
            f'{comp_row["test_quantum_rescue_given_classical_error"]:.4f}',
            "ratio=",
            f'{comp_row["conditional_quantum_rescue_shift_ratio"]:.3f}',
        )

all_model_metrics = pd.concat(model_rows, ignore_index=True)
all_complementarity = pd.DataFrame(complement_rows)
all_controlled_fpr = pd.DataFrame(fpr_rows)
all_runtime = pd.DataFrame(runtime_rows)
all_fusion_weights = pd.DataFrame(fusion_weight_rows)

all_model_metrics.to_csv(
    OUTPUT_ROOT / "all_seed_zero_day_model_metrics.csv", index=False
)
all_complementarity.to_csv(
    OUTPUT_ROOT / "all_seed_complementarity_shift.csv", index=False
)
all_controlled_fpr.to_csv(
    OUTPUT_ROOT / "all_seed_controlled_fpr.csv", index=False
)
all_runtime.to_csv(
    OUTPUT_ROOT / "all_seed_runtime.csv", index=False
)
all_fusion_weights.to_csv(
    OUTPUT_ROOT / "all_seed_static_fusion_weights.csv", index=False
)

print("\nAll 30 locked zero-day runs completed.")


## 10. Aggregate zero-day performance

In [ ]:

main_metrics = [
    "mcc", "balanced_accuracy", "f1", "recall",
    "specificity", "roc_auc", "pr_auc"
]

model_summary = (
    all_model_metrics
    .groupby(["heldout_family", "model"])[main_metrics]
    .agg(["mean", "std", "median"])
)
model_summary.columns = ["_".join(c) for c in model_summary.columns]
model_summary = model_summary.reset_index()

display(model_summary)
model_summary.to_csv(
    OUTPUT_ROOT / "multiseed_zero_day_model_summary.csv", index=False
)

weight_summary = (
    all_fusion_weights
    .groupby("heldout_family")[["classical_weight", "quantum_weight"]]
    .agg(["mean", "std", "median"])
)
weight_summary.columns = ["_".join(c) for c in weight_summary.columns]
weight_summary = weight_summary.reset_index()

display(weight_summary)
weight_summary.to_csv(
    OUTPUT_ROOT / "multiseed_static_weight_summary.csv", index=False
)


## 11. Complementarity-shift summary

In [ ]:

comp_columns = [
    "dev_quantum_only_rate_all",
    "test_quantum_only_rate_all",
    "quantum_only_shift_ratio",
    "dev_quantum_rescue_given_classical_error",
    "test_quantum_rescue_given_classical_error",
    "conditional_quantum_rescue_shift_ratio",
    "dev_classical_only_rate_all",
    "test_classical_only_rate_all",
    "classical_only_shift_ratio",
    "test_oracle_accuracy",
    "test_classical_accuracy",
    "test_quantum_accuracy",
    "oracle_headroom_over_classical",
]

comp_summary = (
    all_complementarity
    .groupby("heldout_family")[comp_columns]
    .agg(["mean", "std", "median"])
)
comp_summary.columns = ["_".join(c) for c in comp_summary.columns]
comp_summary = comp_summary.reset_index()

display(comp_summary)
comp_summary.to_csv(
    OUTPUT_ROOT / "multiseed_complementarity_shift_summary.csv", index=False
)

# Sign-stability: how often did quantum complementarity increase under zero-day shift?
direction_rows = []
for family in LOCKED_ZERO_DAY_CONDITIONS:
    sub = all_complementarity[
        all_complementarity["heldout_family"] == family
    ]
    ratios = sub["conditional_quantum_rescue_shift_ratio"].to_numpy(float)
    direction_rows.append({
        "heldout_family": family,
        "n_seeds": len(sub),
        "seeds_shift_gt_1": int(np.nansum(ratios > 1)),
        "seeds_shift_eq_1": int(np.nansum(np.isclose(ratios, 1))),
        "seeds_shift_lt_1": int(np.nansum(ratios < 1)),
        "median_shift_ratio": float(np.nanmedian(ratios)),
    })

direction_summary = pd.DataFrame(direction_rows)
display(direction_summary)
direction_summary.to_csv(
    OUTPUT_ROOT / "complementarity_shift_direction_stability.csv",
    index=False,
)


## 12. Zero-day recall at development-controlled false-positive rates

In [ ]:

fpr_summary = (
    all_controlled_fpr
    .groupby(["heldout_family", "model", "development_target_fpr"])[
        ["zero_day_recall", "realized_test_fpr", "precision"]
    ]
    .agg(["mean", "std", "median"])
)
fpr_summary.columns = ["_".join(c) for c in fpr_summary.columns]
fpr_summary = fpr_summary.reset_index()

display(fpr_summary)
fpr_summary.to_csv(
    OUTPUT_ROOT / "multiseed_controlled_fpr_summary.csv", index=False
)


## 13. Paired 10-seed statistical comparisons

In [ ]:

paired_rows = []
comparators = [
    "Classical CAT",
    "Quantum CAT",
    "Static Weighted Hybrid",
    "Generic RF Stacking",
]
paired_metrics = ["mcc", "balanced_accuracy", "f1", "recall"]

for family in LOCKED_ZERO_DAY_CONDITIONS:
    subset = all_model_metrics[
        all_model_metrics["heldout_family"] == family
    ]

    hq = subset[
        subset["model"] == "Adaptive HQCA"
    ].set_index("seed")

    for comparator in comparators:
        cp = subset[
            subset["model"] == comparator
        ].set_index("seed")

        common = sorted(set(hq.index) & set(cp.index))

        for metric in paired_metrics:
            a = hq.loc[common, metric].to_numpy(float)
            b = cp.loc[common, metric].to_numpy(float)
            diff = a - b

            if np.any(np.abs(diff) > 1e-12):
                try:
                    stat, p = wilcoxon(
                        a, b,
                        zero_method="wilcox",
                        alternative="two-sided",
                        method="auto",
                    )
                except Exception:
                    stat, p = np.nan, np.nan
            else:
                stat, p = np.nan, 1.0

            paired_rows.append({
                "heldout_family": family,
                "comparison": f"Adaptive HQCA vs {comparator}",
                "metric": metric,
                "n_seeds": len(common),
                "hqca_mean": float(a.mean()),
                "comparator_mean": float(b.mean()),
                "mean_difference": float(diff.mean()),
                "median_difference": float(np.median(diff)),
                "wins": int((diff > 0).sum()),
                "ties": int((np.abs(diff) <= 1e-12).sum()),
                "losses": int((diff < 0).sum()),
                "wilcoxon_statistic": stat,
                "p_value": p,
            })

paired_tests = pd.DataFrame(paired_rows)
display(paired_tests)
paired_tests.to_csv(
    OUTPUT_ROOT / "paired_10seed_model_tests.csv", index=False
)


## 14. Publication figures

In [ ]:

# Figure 1: complementarity shift (conditional quantum rescue).
for family in LOCKED_ZERO_DAY_CONDITIONS:
    sub = all_complementarity[
        all_complementarity["heldout_family"] == family
    ]
    values = [
        sub["dev_quantum_rescue_given_classical_error"].mean(),
        sub["test_quantum_rescue_given_classical_error"].mean(),
    ]
    errors = [
        sub["dev_quantum_rescue_given_classical_error"].std(),
        sub["test_quantum_rescue_given_classical_error"].std(),
    ]

    plt.figure(figsize=(6, 4))
    x = np.arange(2)
    plt.bar(x, values, yerr=errors, capsize=4)
    plt.xticks(x, ["Development OOF", "Zero-day test"])
    plt.ylabel("P(Quantum correct | Classical wrong)")
    plt.title(f"Classical–Quantum complementarity shift — {family}")
    plt.tight_layout()
    plt.savefig(
        OUTPUT_ROOT / f"figure_complementarity_shift_{family.replace(' ', '_')}.png",
        dpi=240,
    )
    plt.show()

# Figure 2: MCC comparison.
plot_models = [
    "Classical CAT",
    "Quantum CAT",
    "Static Weighted Hybrid",
    "Generic RF Stacking",
    "Adaptive HQCA",
]

for family in LOCKED_ZERO_DAY_CONDITIONS:
    sub = all_model_metrics[
        (all_model_metrics["heldout_family"] == family) &
        (all_model_metrics["model"].isin(plot_models))
    ]

    means = [
        sub.loc[sub["model"] == model, "mcc"].mean()
        for model in plot_models
    ]
    stds = [
        sub.loc[sub["model"] == model, "mcc"].std()
        for model in plot_models
    ]

    plt.figure(figsize=(9, 5))
    x = np.arange(len(plot_models))
    plt.bar(x, means, yerr=stds, capsize=4)
    plt.xticks(x, plot_models, rotation=25, ha="right")
    plt.ylabel("MCC")
    plt.title(f"CIC-IDS-2017 zero-day source holdout — {family}")
    plt.tight_layout()
    plt.savefig(
        OUTPUT_ROOT / f"figure_mcc_{family.replace(' ', '_')}.png",
        dpi=240,
    )
    plt.show()

# Figure 3: zero-day recall at 1% development-controlled FPR.
target = 0.01
for family in LOCKED_ZERO_DAY_CONDITIONS:
    sub = all_controlled_fpr[
        (all_controlled_fpr["heldout_family"] == family) &
        (all_controlled_fpr["development_target_fpr"] == target) &
        (all_controlled_fpr["model"].isin(plot_models))
    ]

    means = [
        sub.loc[sub["model"] == model, "zero_day_recall"].mean()
        for model in plot_models
    ]
    stds = [
        sub.loc[sub["model"] == model, "zero_day_recall"].std()
        for model in plot_models
    ]

    plt.figure(figsize=(9, 5))
    x = np.arange(len(plot_models))
    plt.bar(x, means, yerr=stds, capsize=4)
    plt.xticks(x, plot_models, rotation=25, ha="right")
    plt.ylabel("Zero-day attack recall")
    plt.ylim(0, 1)
    plt.title(f"Recall at development-controlled 1% FPR — {family}")
    plt.tight_layout()
    plt.savefig(
        OUTPUT_ROOT / f"figure_recall_at_1pct_fpr_{family.replace(' ', '_')}.png",
        dpi=240,
    )
    plt.show()


## 15. Reproducibility manifest and interpretation guardrails

In [ ]:

manifest = {
    "experiment": "CICIDS2017 zero-day complementarity-shift validation",
    "status": (
        "Pre-specified 10-seed source/attack-family holdout protocol. "
        "CICIDS2017 was used previously in the broader project, but these locked "
        "zero-day source-holdout conditions were not used to develop the router."
    ),
    "dataset_root": str(CIC_DIR),
    "locked_conditions": {
        family: resolved_sources[family].name
        for family in LOCKED_ZERO_DAY_CONDITIONS
    },
    "seeds": SEEDS,
    "per_source_dev_cap": PER_SOURCE_DEV_CAP,
    "max_dev_rows": MAX_DEV_ROWS,
    "max_test_attacks": MAX_TEST_ATTACKS,
    "max_test_benign": MAX_TEST_BENIGN,
    "oof_folds": OOF_FOLDS,
    "meta_folds": META_FOLDS,
    "n_qubits": N_QUBITS,
    "quantum_layers": QUANTUM_LAYERS,
    "quantum_epochs": QUANTUM_EPOCHS,
    "quantum_batch_size": QUANTUM_BATCH_SIZE,
    "fpr_targets": FPR_TARGETS,
    "torch_version": torch.__version__,
    "cuda_build": torch.version.cuda,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_name": GPU_NAME,
    "pennylane_version": qml.__version__,
    "quantum_backend": QUANTUM_BACKEND,
    "heldout_source_is_excluded_from_development": True,
    "heldout_attack_family_removed_from_all_development_sources": True,
    "source_file_used_as_predictor": False,
    "attack_family_used_as_predictor": False,
    "label_used_as_predictor": False,
    "primary_hypothesis": (
        "Classical–Quantum complementarity is attack/distribution dependent; "
        "the rate at which Quantum CAT rescues Classical CAT errors can shift "
        "materially from known development traffic to zero-day attack traffic."
    ),
    "guardrails": [
        "Report all 3 conditions and all 10 seeds.",
        "Do not select only attack families showing increased quantum complementarity.",
        "Do not claim quantum advantage from simulator results.",
        "Do not claim Adaptive HQCA superiority unless supported consistently by effect size and paired tests.",
        "Treat complementarity shift as attack-dependent if direction differs by held-out family.",
    ],
}

(OUTPUT_ROOT / "run_manifest.json").write_text(
    json.dumps(manifest, indent=2, default=str),
    encoding="utf-8",
)

print(json.dumps(manifest, indent=2, default=str))
print("\nZIP and upload this entire results folder when complete:")
print(OUTPUT_ROOT)
